# Chest X-ray pneumonia: the rebuild

Two-stage pipeline on the Kaggle chest X-ray dataset, rebuilt from my Makeen deep learning capstone.

1. **Stage 1** (ResNet50): normal vs pneumonia
2. **Stage 2** (InceptionV3): bacterial vs viral, trained on training-set pneumonia images
3. **Baseline**: one 3-class model, to check that two stages earn their complexity

Before running: **Runtime > Change runtime type > T4 GPU**, and push the `chest-xray-pneumonia-rebuild` folder to GitHub so the next cell can clone it.

In [ ]:
import os
if not os.path.exists("/content/ML-DL"):
    !git clone -q https://github.com/Intisar3/ML-DL.git /content/ML-DL
%cd /content/ML-DL/chest-xray-pneumonia-rebuild

import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none, switch the runtime to GPU")

## 1. Smoke test
Runs the whole pipeline on tiny synthetic images in about a minute. Continue once it prints `SMOKE TEST PASSED`.

In [ ]:
!python -m tests.smoke_test 2>&1 | tail -5

## 2. Download the data
KaggleHub downloads public datasets directly (about 2.3 GB). If it asks for credentials, upload your `kaggle.json` to `~/.kaggle/` and rerun.

In [ ]:
import kagglehub
DATA = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
OUT = "/content/outputs"
print(DATA)

## 3. Patient-grouped split
Train and val are merged and re-split so every patient sits in exactly one split. The official test set stays untouched.

In [ ]:
import pandas as pd
from src.config import Config, TASKS
from src.data import build_index, make_splits

cfg = Config()
train_df, val_df, test_df = make_splits(build_index(DATA), cfg)
summary = pd.DataFrame({name: d["label"].value_counts() for name, d in
                        [("train", train_df), ("val", val_df), ("test", test_df)]}).T
summary["total"] = summary.sum(axis=1)
summary

## 4. Train
Each model trains its new head first, then fine-tunes the top of the backbone at a low learning rate. The best validation weights are saved to `outputs/`.

In [ ]:
from src.train import train_task
_, hist1 = train_task("stage1", DATA, OUT, cfg)

In [ ]:
_, hist2 = train_task("stage2", DATA, OUT, cfg)

In [ ]:
_, hist3 = train_task("three_class", DATA, OUT, cfg)

## 5. Evaluate on the test set
Thresholds are chosen on validation, then applied once to the test set. The two-stage number is end to end: stage 2 receives stage 1's mistakes too.

In [ ]:
import json
from IPython.display import Image, display
from src.evaluate import evaluate_all

results = evaluate_all(DATA, OUT, cfg)
display(pd.DataFrame({k: {m: v for m, v in r.items() if m != "confusion_matrix"}
                      for k, r in results.items()}).T.round(3))
for name in ["stage1_confusion", "stage2_confusion", "two_stage_confusion", "three_class_confusion"]:
    display(Image(f"{OUT}/{name}.png", width=380))

## 6. Grad-CAM
Heatmaps of the regions that drove stage 1's prediction. Useful evidence that the model looks at the lungs.

In [ ]:
import numpy as np
from src.data import make_dataset, task_frame
from src.gradcam import show_gradcam

model1 = keras.models.load_model(f"{OUT}/stage1.keras")
sample = task_frame(test_df, "stage1").groupby("label").sample(2, random_state=cfg.seed)
x, y = next(iter(make_dataset(sample, "stage1", Config(batch_size=len(sample)), training=False)))
probs = model1.predict(x, verbose=0).ravel()
titles = [f"true: {t}\np(pneumonia) = {p:.2f}" for t, p in zip(sample["label"], probs)]
show_gradcam(model1, x.numpy(), "resnet50", titles, path=f"{OUT}/gradcam_stage1.png");

## 7. Save the results
Downloads metrics, figures, and predictions. Copy the numbers into the README table.

In [ ]:
!cd /content && zip -q -r results.zip outputs -x "*.keras"
from google.colab import files
files.download("/content/results.zip")